# Stage 5 — Checking the NSA spatial join

`pipeline/spatial_join.py` gave every located request the `nsa_id` of the neighborhood (NSA) polygon it falls in.

Outputs: `data/interim/neighborhoods.parquet` (279 NSAs, EPSG:4326) and `data/interim/requests_final.parquet` (requests + `nsa_id`). Ids are saved in `reference/nsa_id_map.csv`.

**Read-only:** this notebook only checks results; it changes no files.

In [1]:
import duckdb
import geopandas as gpd
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
N = (ROOT / "data/interim/neighborhoods.parquet").as_posix()
R = (ROOT / "data/interim/requests_final.parquet").as_posix()
con = duckdb.connect()
con.sql("SET TimeZone = 'UTC'")
q = lambda sql: con.sql(sql).df()

## 1. Neighborhoods table
Expect 279 rows, `nsa_id` 1–279 with no gaps or repeats, and coordinates in Baltimore's lon/lat range.

In [2]:
nsa = gpd.read_parquet(N)
print("rows:", len(nsa), "| nsa_id unique:", nsa.nsa_id.is_unique,
      "| range:", nsa.nsa_id.min(), "-", nsa.nsa_id.max(), "| CRS:", nsa.crs.to_epsg())
print("bounds (lon/lat):", nsa.total_bounds.round(4))
print("columns:", len(nsa.columns))
nsa[["nsa_id", "Name", "GlobalID", "Population"]].head()

rows: 279 | nsa_id unique: True | range: 1 - 279 | CRS: 4326
bounds (lon/lat): [-76.7113  39.1972 -76.5297  39.372 ]
columns: 62


,nsa_id,Name,GlobalID,Population
0,1,Abell,{C2454311-47DA-4811-9C3C-7A240845BE5A},975.0
1,2,Allendale,{BA37DF10-E06D-4393-AEBD-F5AD6A818176},3354.0
2,3,Arcadia,{211CB662-043F-47A2-A21D-5ED63AEF0FAB},1077.0
3,4,Arlington,{E02FAE13-0106-43B9-B172-3C50AD3479A2},2163.0
4,5,Armistead Gardens,{849E48A6-070A-4539-9D34-D6F72115E301},3779.0


In [3]:
# Null counts in the demographic columns (Stage 1 found 1 null Population)
nulls = nsa.drop(columns="geometry").isna().sum()
nulls[nulls > 0]

Population              1
White                   1
Blk_AfAm                1
AmInd_AkNa              1
Asian                   1
NatHaw_Pac              1
Other_Race              1
TwoOrMore               1
Hisp_Lat                1
Male                    1
Female                  1
Total_Units             1
Occ_Occupied            1
Occ_Vacant              1
Tenure_Owner            1
Tenure_Renter           1
Vacant_ForRent          1
Vacant_ForSale          1
Vacant_Other_All        1
HH_Total                1
HH_Family               1
HH_Married              1
HH_OtherFamily          1
HH_Male_NoSpouse        1
HH_Female_NoSpouse      1
HH_NonFamily            1
HH_NonFamilyAlone       1
HH_NonFamilyNotAlone    1
HH18_With18             1
HH18_FamilyWith18       1
HH18_NonFamilyWith18    1
HH18_No18               1
HH18_FamilyNo18         1
HH18_NonFamilyNo18      1
Age_U5                  1
Age_5_9                 1
Age_10_14               1
Age_15_17               1
Age_18_21   

## 2. Match rate

In [4]:
q(f'''SELECT nsa_id IS NOT NULL AS in_nsa, count(*) AS n,
            round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
     FROM '{R}' GROUP BY 1 ORDER BY 1 DESC''')

,in_nsa,n,pct
0,True,10567596,97.99
1,False,216884,2.01


## 3. Does `nsa_id` agree with the original `neighborhood` text?
The source data already had a `neighborhood` text column (filled by the city). Where both exist, the polygon we chose should usually carry the same name. Some disagreement is expected near borders and where the city's text is old.

In [5]:
con.register("nsa_names", nsa[["nsa_id", "Name"]])
agree = q(f'''
    SELECT CASE WHEN r.neighborhood IS NULL THEN 'no original text'
                WHEN r.nsa_id IS NULL THEN 'text but no nsa_id'
                WHEN lower(trim(r.neighborhood)) = lower(n.Name) THEN 'agree'
                ELSE 'disagree' END AS result,
           count(*) AS n
    FROM '{R}' r LEFT JOIN nsa_names n USING (nsa_id)
    GROUP BY 1 ORDER BY n DESC''')
agree["pct"] = (100 * agree.n / agree.n.sum()).round(2)
both = agree[agree.result.isin(["agree", "disagree"])]
print(f"agreement where both exist: {100 * both.set_index('result').n.get('agree', 0) / both.n.sum():.2f}%")
agree

agreement where both exist: 96.31%


,result,n,pct
0,agree,10108276,93.73
1,disagree,387487,3.59
2,no original text,288595,2.68
3,text but no nsa_id,122,0.00


In [6]:
# Most common disagreements: original text vs. the polygon's Name
q(f'''SELECT r.neighborhood AS original_text, n.Name AS polygon_name, count(*) AS n
     FROM '{R}' r JOIN nsa_names n USING (nsa_id)
     WHERE r.neighborhood IS NOT NULL AND lower(trim(r.neighborhood)) <> lower(n.Name)
     GROUP BY ALL ORDER BY n DESC LIMIT 15''')

,original_text,polygon_name,n
0,Glenham-Belhar,Hamilton,63352
1,New Southwest/Mount Clare,Mount Clare,42598
2,Parkview/Woodbrook,Auchentoroly-Parkwood,37497
3,Butcher's Hill,Butchers Hill,25183
4,Burleith-Leighton,Parkway,11290
5,Wrenlane,Wilson Park,9403
6,Lower Edmondson Village,Allendale,8067
7,Reisterstown Station,Fallstaff,7784
8,Perkins Homes,Perkins,7068
9,Inner Harbor,Harbor East,6205


## 4. Unmatched points: where are they?
For each request with no `nsa_id`, measure the distance to the nearest NSA polygon (in feet, using the original Maryland State Plane CRS). Points just outside an edge are likely in the harbor or on the city line. Points far away are county addresses or bad coordinates.

In [7]:
pts = q(f'''SELECT longitude, latitude, count(*) AS n FROM '{R}'
           WHERE nsa_id IS NULL GROUP BY ALL''')
g = gpd.GeoDataFrame(pts, geometry=gpd.points_from_xy(pts.longitude, pts.latitude), crs=4326).to_crs(2248)
near = gpd.sjoin_nearest(g, nsa[["Name", "geometry"]].to_crs(2248), distance_col="dist_ft")
near = near.drop_duplicates(subset=["longitude", "latitude"])  # equidistant ties
bins = [0, 165, 1650, 16500, 165000, float("inf")]
labels = ["< 50 m", "50-500 m", "0.5-5 km", "5-50 km", "> 50 km"]
near["distance"] = pd.cut(near.dist_ft, bins=bins, labels=labels, include_lowest=True)
out = near.groupby("distance", observed=False).agg(points=("n", "size"), requests=("n", "sum"))
out["pct_requests"] = (100 * out.requests / out.requests.sum()).round(2)
out

,points,requests,pct_requests
distance,,,
< 50 m,2848,6230,2.87
50-500 m,12982,21965,10.13
0.5-5 km,71504,110430,50.92
5-50 km,51139,78259,36.08
> 50 km,0,0,0.00


In [8]:
# Coordinates that look like placeholders (whole-degree values), not real locations
q(f'''SELECT latitude, longitude, count(*) AS n, min(created_date)::DATE AS first, max(created_date)::DATE AS last
     FROM '{R}'
     WHERE nsa_id IS NULL AND latitude = round(latitude) AND longitude = round(longitude)
     GROUP BY ALL ORDER BY n DESC''')

,latitude,longitude,n,first,last


In [9]:
# Closest unmatched spots with many requests (edge cases worth eyeballing on a map)
near.sort_values("n", ascending=False).query("dist_ft < 1650")[["latitude", "longitude", "n", "Name", "dist_ft"]].head(10).round(1)

,latitude,longitude,n,Name,dist_ft
95859,39.3,-76.6,350,Baltimore Peninsula,490.2
104713,39.4,-76.6,209,North Roland Park/Poplar Hill,443.9
70363,39.4,-76.7,125,Fallstaff,258.9
246,39.3,-76.5,84,Pulaski Industrial Area,0.4
121070,39.4,-76.7,76,Fallstaff,29.2
1012,39.4,-76.6,69,Loch Raven,1202.3
18267,39.3,-76.7,60,Westgate,411.4
70506,39.3,-76.6,50,Baltimore Peninsula,490.2
44249,39.3,-76.7,41,Seton Business Park,539.5
26676,39.4,-76.5,40,Overlea,215.1


## 5. Requests per NSA
Every NSA should get some requests. An NSA with zero would suggest a geometry problem.

In [10]:
per = q(f'''SELECT nsa_id, count(*) AS n FROM '{R}' WHERE nsa_id IS NOT NULL GROUP BY 1''')
per = nsa[["nsa_id", "Name"]].merge(per, on="nsa_id", how="left").fillna({"n": 0})
print("NSAs with zero requests:", (per.n == 0).sum())
pd.concat([per.nlargest(5, "n"), per.nsmallest(5, "n")])

NSAs with zero requests: 0


,nsa_id,Name,n
13,14,Belair-Edison,332217
63,64,Downtown,210864
33,34,Canton,193772
51,52,Coldstream Homestead Montebello,178875
26,27,Broadway East,178787
68,69,Dundalk Marine Terminal,246
238,239,Spring Garden Industrial Area,590
21,22,Blythewood,637
57,58,Curtis Bay Industrial Area,1009
252,253,Villages of Homeland,1076


## Conclusions

- **neighborhoods:** 279 NSAs, `nsa_id` 1–279, reprojected correctly (Baltimore lon/lat bounds). One NSA has all demographics null and 8 lack median age; these come from the source GDB.
- **Match rate ~98%.** No point sat on a shared border (0 ties), and every NSA gets requests.
- **96% agreement** with the city's original `neighborhood` text. The disagreements are mostly renamed or respelled NSAs (Glenham-Belhar → Hamilton, Butcher's Hill → Butchers Hill), not join errors.
- **Unmatched (~2%)** are mostly 0.5–50 km outside the city (county addresses). About 13% are within 500 m of the city edge (harbor or waterfront). All keep `nsa_id` NULL, as decided.
- **Placeholder coordinates** (whole degrees like 39.0, −77.0; about 5.5k rows, mostly 2018) are treated as missing in Stage 3, so the placeholder table above is empty. Most got real coordinates through address matching; the rest moved to `requests_no_location`.